In [ ]:
import os # OS utilities: paths, directory listing, file handling
import numpy as np # Numerical operations and array handling
import polars as pl # Faster DataFrame engine used instead of pandas
from skimage import io, filters, feature, transform  # Image I/O, filters, edge detection, resizing
from skimage.color import rgb2gray # Convert RGB images to grayscale
from skimage.filters import threshold_otsu, threshold_sauvola # Global and adaptive thresholding
from skimage import img_as_ubyte # Convert float images to 8‑bit for saving
import warnings # Suppress non‑critical warnings
from collections import defaultdict # Auto‑initialising dict for class statistics

warnings.filterwarnings("ignore", message=".*low contrast image.*") # Suppress low-contrast warnings

# 1. INPUT + OUTPUT FOLDERS
input_folder = "Waste_Merged" # The merged dataset containing all classes and images
sobel_folder = "Edges_Sobel" # Output folder for Sobel edge maps
canny_folder = "Edges_Canny" # Output folder for Canny edge maps
otsu_folder = "Otsu_Threshold" # Output folder for Otsu masks
adaptive_folder = "Adaptive_Threshold" # Output folder for adaptive masks
masked_folder = "Masked_Images" # Output folder for masked RGB images

# Create output folders if they don't already exist
for f in [sobel_folder, canny_folder, otsu_folder, adaptive_folder, masked_folder]:
    os.makedirs(f, exist_ok=True)

valid_ext = (".jpg", ".jpeg", ".png") # Allowed image extensions

# 2. RESUME CHECKPOINT SYSTEM
processed_log = "processed_images.txt" # File storing names of already processed images

# Load previously processed filenames
if os.path.exists(processed_log):
    with open(processed_log, "r") as f:
        processed_set = set(line.strip() for line in f)
else:
    processed_set = set()

# 3. STORAGE FOR RESULTS
image_results = [] # Per-image feature results
class_stats = defaultdict(list) # Per-class aggregated statistics

# Progress tracking
all_images = [f for f in os.listdir(input_folder) if f.lower().endswith(valid_ext)] # All valid images
total = len(all_images) # Total number of images
count = 0 # Progress counter

# 4. PROCESS EACH IMAGE
for filename in all_images:

    if filename in processed_set: # Skip images already processed
        continue

    count += 1  # Update progress counter
    print(f"[{count}/{total}] Processing: {filename}") # Progress display

    img_path = os.path.join(input_folder, filename) # Full path to image
    label = filename.split("_")[0] # Extract class label from filename

    # Read image safely
    try:
        image = io.imread(img_path)
    except Exception as e:
        print(f"Skipping corrupted file: {filename} — {e}")
        continue

    # Resize to 224×224 (CNN/ViT standard)
    try:
        image = transform.resize(image, (224, 224), anti_aliasing=True)
    except Exception as e:
        print(f"Resize failed for {filename}: {e}")
        continue

    gray = rgb2gray(image) # Convert to grayscale

    # EDGE DETECTION
    sobel_edges = filters.sobel(gray) # Sobel operator
    canny_edges = feature.canny(gray, sigma=1.5) # Canny detector

    # THRESHOLDING (MASK CREATION)
    t = threshold_otsu(gray) # Global threshold
    otsu_mask = gray > t

    sauvola_t = threshold_sauvola(gray, window_size=25) # Local threshold
    adaptive_mask = gray > sauvola_t

    final_mask = adaptive_mask # Adaptive mask chosen as default

    # APPLY MASK TO ORIGINAL RGB
    masked_rgb = image.copy() # Copy original image
    masked_rgb[~final_mask] = 0 # Remove background pixels

    # BRIGHTNESS + COLOUR ANALYSIS
    fg_pixels = masked_rgb[final_mask] # Foreground pixels only

    if fg_pixels.size == 0: # Skip empty masks
        print(f"No foreground detected in {filename}")
        continue

    brightness = gray[final_mask].mean() # Mean grayscale brightness
    R = fg_pixels[:, 0].mean() # Mean red channel
    G = fg_pixels[:, 1].mean() # Mean green channel
    B = fg_pixels[:, 2].mean() # Mean blue channel
    saturation = (fg_pixels.max(axis=1) - fg_pixels.min(axis=1)).mean() # Channel spread

    # STORE PER-IMAGE RESULTS
    image_results.append({
        "filename": filename,
        "label": label,
        "brightness": brightness,
        "mean_R": R,
        "mean_G": G,
        "mean_B": B,
        "saturation": saturation
    })

    class_stats[label].append([brightness, R, G, B, saturation]) # Add to class summary

    # SAVE OUTPUT IMAGES
    base = os.path.splitext(filename)[0] # Remove extension

    io.imsave(os.path.join(sobel_folder, f"{base}_sobel.png"), img_as_ubyte(sobel_edges))
    io.imsave(os.path.join(canny_folder, f"{base}_canny.png"), img_as_ubyte(canny_edges))
    io.imsave(os.path.join(otsu_folder, f"{base}_otsu.png"), img_as_ubyte(otsu_mask))
    io.imsave(os.path.join(adaptive_folder, f"{base}_adaptive.png"), img_as_ubyte(adaptive_mask))
    io.imsave(os.path.join(masked_folder, f"{base}_masked.png"), img_as_ubyte(masked_rgb))

    # UPDATE CHECKPOINT LOG
    with open(processed_log, "a") as f:
        f.write(filename + "\n") # Mark image as processed

# 5. SAVE PER-IMAGE CSV (POLARS)
df = pl.DataFrame(image_results) # Convert to Polars DataFrame
df.write_csv("brightness_colour_analysis.csv") # Save CSV
print("\nSaved: brightness_colour_analysis.csv")

print("\nProcessing complete. Run the next cell for class summary statistics.")

[1/42832] Processing: battery_original_000000.jpg
[2/42832] Processing: battery_original_000001.jpg
[3/42832] Processing: battery_original_000002.jpg
[4/42832] Processing: battery_original_000003.jpg
[5/42832] Processing: battery_original_000004.jpg
[6/42832] Processing: battery_original_000005.jpg
[7/42832] Processing: battery_original_000006.jpg
[8/42832] Processing: battery_original_000007.jpg
[9/42832] Processing: battery_original_000008.jpg
[10/42832] Processing: battery_original_000009.jpg
[11/42832] Processing: battery_original_000010.jpg
[12/42832] Processing: battery_original_000011.jpg
[13/42832] Processing: battery_original_000012.jpg
[14/42832] Processing: battery_original_000013.jpg
[15/42832] Processing: battery_original_000014.jpg
[16/42832] Processing: battery_original_000015.jpg
[17/42832] Processing: battery_original_000016.jpg
[18/42832] Processing: battery_original_000017.jpg
[19/42832] Processing: battery_original_000018.jpg
[20/42832] Processing: battery_original_

In [1]:
import polars as pl # Load Polars to read the CSV
import numpy as np # Needed for array operations

# Load the per-image CSV generated in the main pipeline
df = pl.read_csv("brightness_colour_analysis.csv") # Read CSV into Polars DataFrame

# Rebuild class_stats dictionary from the CSV
class_stats = {} # Dictionary: class_label → numpy array of feature rows

for label in df["label"].unique(): # Loop through each class
    subset = df.filter(pl.col("label") == label) # Filter rows for this class
    
    # Convert relevant columns into a NumPy array
    arr = np.column_stack([
        subset["brightness"],
        subset["mean_R"],
        subset["mean_G"],
        subset["mean_B"],
        subset["saturation"]
    ])
    
    class_stats[label] = arr # Store in dictionary

# Print class summary statistics
print("\n================ CLASS SUMMARY STATISTICS ================\n")

for cls, arr in class_stats.items():
    print(f"Class: {cls}")
    print(f"  Count: {len(arr)}")
    print(f"  Mean Brightness: {arr[:,0].mean():.4f}")
    print(f"  Mean R: {arr[:,1].mean():.2f}")
    print(f"  Mean G: {arr[:,2].mean():.2f}")
    print(f"  Mean B: {arr[:,3].mean():.2f}")
    print(f"  Mean Saturation: {arr[:,4].mean():.4f}")
    print("--------------------------------------------------------")


================ CLASS SUMMARY STATISTICS ================

Class: textile
  Count: 10023
  Mean Brightness: 0.5915
  Mean R: 0.63
  Mean G: 0.58
  Mean B: 0.56
  Mean Saturation: 0.1089
--------------------------------------------------------
Class: plastic
  Count: 6194
  Mean Brightness: 0.6223
  Mean R: 0.62
  Mean G: 0.62
  Mean B: 0.62
  Mean Saturation: 0.0923
--------------------------------------------------------
Class: metal
  Count: 3990
  Mean Brightness: 0.6571
  Mean R: 0.67
  Mean G: 0.66
  Mean B: 0.64
  Mean Saturation: 0.0722
--------------------------------------------------------
Class: organic
  Count: 2533
  Mean Brightness: 0.6051
  Mean R: 0.66
  Mean G: 0.60
  Mean B: 0.51
  Mean Saturation: 0.1734
--------------------------------------------------------
Class: battery
  Count: 2268
  Mean Brightness: 0.8083
  Mean R: 0.81
  Mean G: 0.81
  Mean B: 0.80
  Mean Saturation: 0.0379
--------------------------------------------------------
Class: glass
  Count: 612